In [ ]:
!pip install -q kaggle

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR']= '/content'

In [ ]:
!kaggle datasets download -d disisbig/telugu-wikipedia-articles

100% 199M/199M [00:01<00:00, 180MB/s]
100% 199M/199M [00:01<00:00, 179MB/s]


In [2]:
!unzip /content/data.zip -d dataset

Archive:  /content/data.zip
   creating: dataset/data/train/
  inflating: dataset/data/train/0.txt  
  inflating: dataset/data/train/1.txt  
  inflating: dataset/data/train/10.txt  
 extracting: dataset/data/train/101.txt  
  inflating: dataset/data/train/102.txt  
  inflating: dataset/data/train/104.txt  
  inflating: dataset/data/train/105.txt  
  inflating: dataset/data/train/106.txt  
  inflating: dataset/data/train/107.txt  
  inflating: dataset/data/train/108.txt  
  inflating: dataset/data/train/109.txt  
  inflating: dataset/data/train/11.txt  
  inflating: dataset/data/train/110.txt  
  inflating: dataset/data/train/111.txt  
  inflating: dataset/data/train/112.txt  
  inflating: dataset/data/train/113.txt  
  inflating: dataset/data/train/114.txt  
  inflating: dataset/data/train/116.txt  
  inflating: dataset/data/train/12.txt  
  inflating: dataset/data/train/13.txt  
  inflating: dataset/data/train/14.txt  
  inflating: dataset/data/train/15.txt  
  inflating: dataset/data

In [3]:
import os
import nltk
import torch
import numpy as np
from transformers import XLNetTokenizer, XLNetModel
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK data if not already downloaded
nltk.download('punkt')

# Initialize XLNet tokenizer and model
tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")
model = XLNetModel.from_pretrained("xlnet-base-cased")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/467M [00:00<?, ?B/s]

In [4]:
# Function to tokenize and encode text
def tokenize_and_encode(text):
    tokens = tokenizer.encode(text, add_special_tokens=True, max_length=512)
    return tokens

# Function to generate sentence embeddings using XLNet
def generate_sentence_embeddings(sentences):
    embeddings = []
    for sentence in sentences:
        token_ids = tokenize_and_encode(sentence)
        with torch.no_grad():
            outputs = model(torch.tensor(token_ids).unsqueeze(0))
        embeddings.append(torch.mean(outputs.last_hidden_state, dim=1).squeeze().numpy())
    return embeddings

# Function to calculate cosine similarity between sentence embeddings
def calculate_similarity_matrix(embeddings):
    similarity_matrix = cosine_similarity(embeddings)
    return similarity_matrix


In [5]:

# Function to generate extractive summary
def generate_extractive_summary(text, num_sentences=3):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text
    embeddings = generate_sentence_embeddings(sentences)
    similarity_matrix = calculate_similarity_matrix(embeddings)
    # Initialize score for each sentence
    scores = np.zeros(len(sentences))
    # Calculate score for each sentence based on similarity with other sentences
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                scores[i] += similarity_matrix[i][j]
    # Get indices of top scoring sentences
    top_sentences_indices = np.argsort(scores)[::-1][:num_sentences]
    # Sort indices to maintain original order
    top_sentences_indices = sorted(top_sentences_indices)
    # Generate extractive summary
    summary = ' '.join([sentences[i] for i in top_sentences_indices])
    return summary

In [6]:
# Path to the directory containing text files
data_path = "/content/dataset/data/train"

# Function to read text files from directory
def read_text_files(directory):
    texts = []
    for filename in os.listdir(directory):
        with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
            texts.append(file.read())
    return texts


In [7]:
# Read text data from directory
texts = read_text_files(data_path)

# Generate summaries for each text
for i, text in enumerate(texts):
    summary = generate_extractive_summary(text)
    print(f"Summary for text {i+1}:\n{summary}\n")


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Summary for text 1:
తూర్పు గోదావరిజిల్లా లోని 19 శాసనసభ నియోజకవర్గాలలో రాజమహేంద్రవరం పట్టణ శాసనసభ నియోజకవర్గం ఒకటి. ఇంతవరకు సంవత్సరాల వారీగా నియోజకవర్గంలో గెలుపొందిన సభ్యుల పూర్తి వివరాలు ఈ క్రింది పట్టికలో నుదహరించబడినవి. అనంతపురం · వైఎస్ఆర్ · కర్నూలు · కృష్ణా · గుంటూరు · చిత్తూరు · తూర్పు గోదావరి · శ్రీ.పొ.శ్రీ నెల్లూరు · పశ్చిమ గోదావరి · ప్రకాశం · విజయనగరం · విశాఖపట్నం · శ్రీకాకుళం ·

Summary for text 2:
మరికొన్ని ఇటువంటి పేరులు గల వ్యాసాల కోసం సీతా కళ్యాణం అయోమయ నివృత్తి పేజీ కూడా చూడండి. హరినాథ్ గీతాంజలి శ్రీరామ, సీత పాత్రలు ధరించారు. ఆ సమయంలో ఆయనని రామారావు సంగీత దర్శకత్వానికి ఒప్పించి ఈ సినిమాకు అజరామరమైన గీతాలు చేయించుకున్నారు.

Summary for text 3:
ఎమ్బీబీయెస్ కాకుండా ఇతర డిగ్రీ చదివిన డాక్టర్లు ముగ్గురు ఉన్నారు. సమీప గ్రామాల నుండి ఆటో సౌకర్యం కూడా ఉంది. గ్రామంలో గృహావసరాల నిమిత్తం విద్యుత్ సరఫరా వ్యవస్థ ఉంది.

Summary for text 4:
ఆమె వివిధ చర్చిల్లో గాస్పెల్ గాయనిగా కూడా ప్రసిద్ధి చెందింది. ఆమె తల్లి సుశీల సారధి ప్రముఖ మద్రాసు గాయక బృందాన్ని నడిపేది. ఆమె పలు చర్చిల్లో పాటలు పా

In [8]:
import os
import nltk
import torch
import numpy as np
from transformers import XLNetTokenizer, XLNetModel
from sklearn.metrics.pairwise import cosine_similarity



In [9]:

# Function to tokenize and encode Telugu text
def tokenize_and_encode(text):
    tokens = tokenizer.encode(text, add_special_tokens=True, max_length=512)
    return tokens

# Function to generate sentence embeddings using XLNet
def generate_sentence_embeddings(sentences):
    embeddings = []
    for sentence in sentences:
        token_ids = tokenize_and_encode(sentence)
        with torch.no_grad():
            outputs = model(torch.tensor(token_ids).unsqueeze(0))
        embeddings.append(torch.mean(outputs.last_hidden_state, dim=1).squeeze().numpy())
    return embeddings


In [10]:
# Function to calculate cosine similarity between sentence embeddings
def calculate_similarity_matrix(embeddings):
    similarity_matrix = cosine_similarity(embeddings)
    return similarity_matrix

# Function to generate extractive summary from Telugu text using XLNet
def generate_extractive_summary_xlnet(text, num_sentences=3):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text
    embeddings = generate_sentence_embeddings(sentences)
    similarity_matrix = calculate_similarity_matrix(embeddings)
    # Initialize score for each sentence
    scores = np.zeros(len(sentences))
    # Calculate score for each sentence based on similarity with other sentences
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                scores[i] += similarity_matrix[i][j]
    # Get indices of top scoring sentences
    top_sentences_indices = np.argsort(scores)[::-1][:num_sentences]
    # Sort indices to maintain original order
    top_sentences_indices = sorted(top_sentences_indices)
    # Generate extractive summary
    summary = ' '.join([sentences[i] for i in top_sentences_indices])
    return summary


In [11]:
# Example Telugu text for summarization
telugu_text = "సర్వజ్ఞాత్మక వివరణలు అందుకున్న సాధారణ ప్రముఖులు, వేతనాల వివరాలు, రాజకీయ మార్గదర్శకులు, కార్పొరేట్ మేమ్బర్లు, పరిశ్రమ ప్రణాళికలు, జాబానికి సంబంధించిన పరీక్షలు, విభిన్న విద్యాలయాలు, విద్యార్థులు, ప్రముఖులు మరియు స్కూల్ సమాచారం ఉన్నాయి."

# Generate extractive summary using XLNet
summary_xlnet = generate_extractive_summary_xlnet(telugu_text)
print("Extractive Summary (XLNet):")
print(summary_xlnet)


Extractive Summary (XLNet):
సర్వజ్ఞాత్మక వివరణలు అందుకున్న సాధారణ ప్రముఖులు, వేతనాల వివరాలు, రాజకీయ మార్గదర్శకులు, కార్పొరేట్ మేమ్బర్లు, పరిశ్రమ ప్రణాళికలు, జాబానికి సంబంధించిన పరీక్షలు, విభిన్న విద్యాలయాలు, విద్యార్థులు, ప్రముఖులు మరియు స్కూల్ సమాచారం ఉన్నాయి.


In [12]:
import nltk
import torch
import numpy as np
from transformers import XLNetTokenizer, XLNetModel
from sklearn.metrics.pairwise import cosine_similarity

# Path to the sample text file
sample_file_path = "/content/dataset/data/valid/123.txt"

# Read the sample text file
with open(sample_file_path, 'r', encoding='utf-8') as file:
    sample_text = file.read()

# Generate summary for the sample text
summary = generate_extractive_summary(sample_text)
print("Summary for the input text file:\n", summary)


Summary for the input text file:
 ఈ ప్రమాదాల వలన ఎంతో ప్రాణ నష్టం మరియు ఆస్తి నష్టం సంభవిస్తుంది. దీపావళి పండగలో కాల్చే బాణాసంచా మూలంగా ఇంట్లో సామాన్యంగా అగ్ని ప్రమాదాలు జరుగుతాయి. ఎలాంటి అత్యవసర సమయాల్లోనైనా ఆ శాఖ నుంచి సేవలు విధంగా ఆదేశాలు జారీ చేశారు.


In [16]:
!pip install rouge
import rouge  # fixed: was "import rogue" (typo)

In [17]:
from rouge import Rouge

In [18]:
import nltk
import numpy as np
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge
from transformers import XLNetTokenizer, XLNetModel


In [19]:

# Inference function to generate summaries using Telugu XLNet model
def generate_summary(text, num_sentences=3):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text
    embeddings = generate_sentence_embeddings(sentences)
    similarity_matrix = calculate_similarity_matrix(embeddings)
    # Initialize score for each sentence
    scores = np.zeros(len(sentences))
    # Calculate score for each sentence based on similarity with other sentences
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                scores[i] += similarity_matrix[i][j]
    # Get indices of top scoring sentences
    top_sentences_indices = np.argsort(scores)[::-1][:num_sentences]
    # Sort indices to maintain original order
    top_sentences_indices = sorted(top_sentences_indices)
    # Generate extractive summary
    summary = ' '.join([sentences[i] for i in top_sentences_indices])
    return summary

In [20]:
# Example Telugu text for evaluation
reference_text = "ఈ మండల కేంద్రమైన ముంచింగిపుట్టు నుండి 4 కి. మీ. దూరం లోను, సమీప పట్టణమైన జైపూరు నుండి 134 కి. మీ. దూరంలోనూ ఉంది."

# Generate summary using inference function
generated_summary = generate_summary(reference_text)

# Function to calculate BLEU score
def calculate_bleu(reference, summary):
    reference_tokens = nltk.word_tokenize(reference.lower())
    summary_tokens = nltk.word_tokenize(summary.lower())
    return sentence_bleu([reference_tokens], summary_tokens)

In [21]:
# Function to calculate ROUGE score
def calculate_rouge(reference, summary):
    rouge = Rouge()
    scores = rouge.get_scores(summary, reference)
    rouge_1_recall = scores[0]['rouge-1']['r']
    rouge_1_precision = scores[0]['rouge-1']['p']
    rouge_1_f1 = scores[0]['rouge-1']['f']
    return rouge_1_recall, rouge_1_precision, rouge_1_f1

# Calculate BLEU score
bleu_score = calculate_bleu(reference_text, generated_summary)
print("BLEU Score:", bleu_score)

# Calculate ROUGE score
rouge_recall, rouge_precision, rouge_f1 = calculate_rouge(reference_text, generated_summary)
print("ROUGE-1 Recall:", rouge_recall)
print("ROUGE-1 Precision:", rouge_precision)
print("ROUGE-1 F1 Score:", rouge_f1)


BLEU Score: 0.06726367218011521
ROUGE-1 Recall: 0.1875
ROUGE-1 Precision: 1.0
ROUGE-1 F1 Score: 0.31578947102493077
